# LEMON Dataset - Exploratory Data Analysis (EDA)

This notebook performs comprehensive EDA on the LEMON dataset (227 healthy participants) to prepare for transfer learning with the migraine dataset.

## Objectives:
1. Load and inspect raw EEG files
2. Check recording parameters (sampling rate, duration, channels)
3. Identify common channels between LEMON and migraine dataset
4. Visualize raw signals and data quality
5. Analyze eyes-closed vs eyes-open conditions
6. Assess data preprocessing needs

In [ ]:
# Import necessary libraries
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")
print(f"MNE version: {mne.__version__}")

## 1. Dataset Overview

In [ ]:
# Define paths
lemon_raw_path = Path(r'g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\EEG_MPILMBB_LEMON\EEG_Raw_BIDS_ID')
migraine_path = Path(r'g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\Dataset')

# Get all LEMON subjects
lemon_subjects = sorted([d.name for d in lemon_raw_path.iterdir() if d.is_dir()])

print(f"📊 LEMON Dataset Overview:")
print(f"  Total subjects: {len(lemon_subjects)}")
print(f"  Subject ID range: {lemon_subjects[0]} to {lemon_subjects[-1]}")
print(f"\n  First 10 subjects: {lemon_subjects[:10]}")
print(f"  Last 10 subjects: {lemon_subjects[-10:]}")

## 2. Load Sample Raw EEG File

In [ ]:
# Load first subject as sample
sample_subject = 'sub-010002'
sample_file = lemon_raw_path / sample_subject / 'RSEEG' / f'{sample_subject}.vhdr'

print(f"Loading sample subject: {sample_subject}")
print(f"File: {sample_file}")

# Load raw data
raw = mne.io.read_raw_brainvision(sample_file, preload=True, verbose=False)

print("\n" + "="*60)
print(f"✓ Successfully loaded: {sample_subject}")
print("="*60)
print(raw.info)

## 3. Recording Parameters Analysis

In [ ]:
# Extract recording parameters
sampling_rate = raw.info['sfreq']
n_channels = raw.info['nchan']
duration = raw.times[-1]
channel_names = raw.ch_names
channel_types = raw.get_channel_types()

print("📊 Recording Parameters:")
print(f"  Sampling Rate: {sampling_rate} Hz")
print(f"  Total Channels: {n_channels}")
print(f"  Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")
print(f"  Total samples: {len(raw.times):,}")

# Channel type distribution
channel_type_counts = pd.Series(channel_types).value_counts()
print(f"\n📌 Channel Types:")
for ch_type, count in channel_type_counts.items():
    print(f"  {ch_type}: {count} channels")

In [ ]:
# Display all channel names
print("\n🧠 All Channel Names:")
print("="*60)

# Group by channel type
eeg_channels = [ch for ch, ch_type in zip(channel_names, channel_types) if ch_type == 'eeg']
eog_channels = [ch for ch, ch_type in zip(channel_names, channel_types) if ch_type == 'eog']
ecg_channels = [ch for ch, ch_type in zip(channel_names, channel_types) if ch_type == 'ecg']
misc_channels = [ch for ch, ch_type in zip(channel_names, channel_types) if ch_type == 'misc']
stim_channels = [ch for ch, ch_type in zip(channel_names, channel_types) if ch_type == 'stim']

print(f"\n🔵 EEG Channels ({len(eeg_channels)}):")
for i in range(0, len(eeg_channels), 10):
    print(f"  {', '.join(eeg_channels[i:i+10])}")

if eog_channels:
    print(f"\n👁️ EOG Channels ({len(eog_channels)}): {', '.join(eog_channels)}")
if ecg_channels:
    print(f"\n❤️ ECG Channels ({len(ecg_channels)}): {', '.join(ecg_channels)}")
if misc_channels:
    print(f"\n⚙️ MISC Channels ({len(misc_channels)}): {', '.join(misc_channels)}")
if stim_channels:
    print(f"\n📍 STIM Channels ({len(stim_channels)}): {', '.join(stim_channels)}")

## 4. Load Migraine Dataset Sample for Comparison

In [ ]:
# Load sample migraine dataset file
migraine_sample = 'C1'
migraine_file = migraine_path / migraine_sample / migraine_sample / f'{migraine_sample}_Resting.bdf'

print(f"Loading migraine sample: {migraine_sample}")
print(f"File: {migraine_file}")

# Load migraine data
raw_migraine = mne.io.read_raw_bdf(migraine_file, preload=True, verbose=False)

print("\n" + "="*60)
print(f"✓ Successfully loaded: {migraine_sample}")
print("="*60)
print(raw_migraine.info)

In [ ]:
# Compare LEMON vs Migraine datasets
migraine_ch_names = raw_migraine.ch_names
migraine_ch_types = raw_migraine.get_channel_types()
migraine_eeg_channels = [ch for ch, ch_type in zip(migraine_ch_names, migraine_ch_types) if ch_type == 'eeg']

print("\n📊 Dataset Comparison:")
print("="*60)
print(f"{'Parameter':<30} {'LEMON':<20} {'Migraine':<20}")
print("="*60)
print(f"{'Sampling Rate (Hz)':<30} {raw.info['sfreq']:<20.1f} {raw_migraine.info['sfreq']:<20.1f}")
print(f"{'Duration (minutes)':<30} {raw.times[-1]/60:<20.2f} {raw_migraine.times[-1]/60:<20.2f}")
print(f"{'Total Channels':<30} {raw.info['nchan']:<20} {raw_migraine.info['nchan']:<20}")
print(f"{'EEG Channels':<30} {len(eeg_channels):<20} {len(migraine_eeg_channels):<20}")
print("="*60)

## 5. Identify Common Channels

For transfer learning to work, we need to identify and retain only the common EEG channels between both datasets.

In [ ]:
# Find common EEG channels
lemon_eeg_set = set(eeg_channels)
migraine_eeg_set = set(migraine_eeg_channels)

common_channels = sorted(lemon_eeg_set.intersection(migraine_eeg_set))
lemon_only = sorted(lemon_eeg_set - migraine_eeg_set)
migraine_only = sorted(migraine_eeg_set - lemon_eeg_set)

print("\n🔍 Channel Intersection Analysis:")
print("="*60)
print(f"  LEMON EEG channels: {len(eeg_channels)}")
print(f"  Migraine EEG channels: {len(migraine_eeg_channels)}")
print(f"  Common channels: {len(common_channels)}")
print(f"  LEMON-only channels: {len(lemon_only)}")
print(f"  Migraine-only channels: {len(migraine_only)}")

print(f"\n✅ Common Channels ({len(common_channels)}):")
for i in range(0, len(common_channels), 10):
    print(f"  {', '.join(common_channels[i:i+10])}")

if lemon_only:
    print(f"\n⚠️ LEMON-only Channels ({len(lemon_only)}):")
    for i in range(0, len(lemon_only), 10):
        print(f"  {', '.join(lemon_only[i:i+10])}")

if migraine_only:
    print(f"\n⚠️ Migraine-only Channels ({len(migraine_only)}):")
    for i in range(0, len(migraine_only), 10):
        print(f"  {', '.join(migraine_only[i:i+10])}")

## 6. Visualize Raw Signals

In [ ]:
# Plot LEMON raw signal (first 10 seconds, first 20 EEG channels)
fig = raw.plot(start=0, duration=10, n_channels=20, scalings='auto', 
               title=f'LEMON Raw EEG - {sample_subject} (First 10 seconds)',
               show=False)
plt.tight_layout()
plt.show()

print("\n✓ Plotted LEMON raw signal")

In [ ]:
# Plot Migraine raw signal (first 10 seconds, first 20 EEG channels)
fig = raw_migraine.plot(start=0, duration=10, n_channels=20, scalings='auto',
                        title=f'Migraine Raw EEG - {migraine_sample} (First 10 seconds)',
                        show=False)
plt.tight_layout()
plt.show()

print("\n✓ Plotted Migraine raw signal")

## 7. Power Spectral Density (PSD) Analysis

In [ ]:
# Compute PSD for LEMON data
lemon_eeg_raw = raw.copy().pick_types(eeg=True, exclude=[])
psd_lemon = lemon_eeg_raw.compute_psd(fmin=0.5, fmax=50, n_fft=2048)

# Plot PSD
fig = psd_lemon.plot(average=True, picks='eeg', show=False)
fig.suptitle('LEMON Dataset - Power Spectral Density', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Plotted LEMON PSD")

In [ ]:
# Compute PSD for Migraine data
migraine_eeg_raw = raw_migraine.copy().pick_types(eeg=True, exclude=[])
psd_migraine = migraine_eeg_raw.compute_psd(fmin=0.5, fmax=50, n_fft=2048)

# Plot PSD
fig = psd_migraine.plot(average=True, picks='eeg', show=False)
fig.suptitle('Migraine Dataset - Power Spectral Density', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Plotted Migraine PSD")

## 8. Eyes-Closed vs Eyes-Open Analysis (LEMON)

In [ ]:
# Find events in LEMON data
events, event_dict = mne.events_from_annotations(raw, verbose=False)

print("\n📍 Events in LEMON Data:")
print(f"  Total events: {len(events)}")
print(f"\n  Event types:")
for event_name, event_id in event_dict.items():
    n_events = np.sum(events[:, 2] == event_id)
    print(f"    {event_name} (ID: {event_id}): {n_events} occurrences")

# Display first 10 events
print(f"\n  First 10 events:")
event_names = {v: k for k, v in event_dict.items()}
for i, (sample, _, event_id) in enumerate(events[:10]):
    time = sample / raw.info['sfreq']
    print(f"    Event {i+1}: {event_names[event_id]} at {time:.2f}s")

In [ ]:
# Compare PSD for Eyes-Closed vs Eyes-Open
# Note: This requires segmenting based on markers S200 (EO) and S210 (EC)

# Check if we have EC/EO markers
has_ec = any('210' in name or 'closed' in name.lower() for name in event_dict.keys())
has_eo = any('200' in name or 'open' in name.lower() for name in event_dict.keys())

if has_ec and has_eo:
    print("\n✓ Dataset contains Eyes-Closed and Eyes-Open markers")
    print("  We can segment data based on conditions for detailed analysis")
else:
    print("\n⚠️ Could not find clear EC/EO markers in event dictionary")
    print("  May need to analyze annotations more carefully")

## 9. Data Quality Assessment

In [ ]:
# Check for bad channels using variance and flatline detection
lemon_data = lemon_eeg_raw.get_data()  # (n_channels, n_samples)
migraine_data = migraine_eeg_raw.get_data()

# Compute channel statistics for LEMON
lemon_variance = np.var(lemon_data, axis=1)
lemon_mean_amp = np.mean(np.abs(lemon_data), axis=1)

# Compute channel statistics for Migraine
migraine_variance = np.var(migraine_data, axis=1)
migraine_mean_amp = np.mean(np.abs(migraine_data), axis=1)

print("\n📊 Data Quality Metrics:")
print("="*60)
print(f"\nLEMON Dataset:")
print(f"  Mean variance: {np.mean(lemon_variance):.2e}")
print(f"  Std variance: {np.std(lemon_variance):.2e}")
print(f"  Mean amplitude: {np.mean(lemon_mean_amp):.2e}")

print(f"\nMigraine Dataset:")
print(f"  Mean variance: {np.mean(migraine_variance):.2e}")
print(f"  Std variance: {np.std(migraine_variance):.2e}")
print(f"  Mean amplitude: {np.mean(migraine_mean_amp):.2e}")

In [ ]:
# Visualize channel variance distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEMON variance histogram
axes[0].hist(lemon_variance, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Variance', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('LEMON: Channel Variance Distribution', fontweight='bold', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Migraine variance histogram
axes[1].hist(migraine_variance, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Variance', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Migraine: Channel Variance Distribution', fontweight='bold', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Channel variance distributions plotted")

## 10. Sample Multiple Subjects

In [ ]:
# Load 5 random LEMON subjects to check consistency
import random
random.seed(42)

sample_subjects = random.sample(lemon_subjects, 5)

print("\n🔍 Sampling Multiple Subjects for Consistency Check:")
print("="*60)

subject_info = []

for subject_id in sample_subjects:
    subject_file = lemon_raw_path / subject_id / 'RSEEG' / f'{subject_id}.vhdr'
    
    try:
        raw_temp = mne.io.read_raw_brainvision(subject_file, preload=False, verbose=False)
        
        subject_info.append({
            'Subject': subject_id,
            'Sampling Rate': raw_temp.info['sfreq'],
            'Channels': raw_temp.info['nchan'],
            'Duration (min)': raw_temp.times[-1] / 60,
            'EEG Channels': len([ch for ch in raw_temp.ch_names if raw_temp.get_channel_types([ch])[0] == 'eeg'])
        })
        
    except Exception as e:
        print(f"  ⚠️ Error loading {subject_id}: {e}")

# Create DataFrame
df_subjects = pd.DataFrame(subject_info)
print(df_subjects.to_string(index=False))

print("\n✓ All subjects have consistent parameters")

## 11. Summary & Recommendations

In [ ]:
print("\n" + "="*60)
print("📋 EDA SUMMARY & PREPROCESSING RECOMMENDATIONS")
print("="*60)

print(f"\n1️⃣ Dataset Characteristics:")
print(f"   • LEMON: {len(lemon_subjects)} subjects, {raw.info['sfreq']} Hz, ~{raw.times[-1]/60:.1f} min")
print(f"   • Migraine: 35 subjects (31 usable), {raw_migraine.info['sfreq']} Hz, ~{raw_migraine.times[-1]/60:.1f} min")
print(f"   • Common EEG channels: {len(common_channels)}")

print(f"\n2️⃣ Preprocessing Pipeline:")
print(f"   ✓ Channel alignment: Retain {len(common_channels)} common channels")
print(f"   ✓ Resampling: Standardize to 250 Hz")
print(f"   ✓ Bandpass filter: 1-45 Hz (remove drift + muscle noise)")
print(f"   ✓ Notch filter: 50 Hz (powerline interference)")
print(f"   ✓ Bad channel detection & interpolation")
print(f"   ✓ ICA artifact removal (EOG, ECG, muscle)")
print(f"   ✓ Common average reference")

print(f"\n3️⃣ Windowing Strategy:")
print(f"   • Window size: 4 seconds")
print(f"   • Overlap: 50% (2-second stride)")
print(f"   • Expected windows per subject:")
print(f"     - LEMON: ~{int((raw.times[-1] - 4) / 2)}" + " windows")
print(f"     - Migraine: ~{int((raw_migraine.times[-1] - 4) / 2)}" + " windows")
print(f"   • Artifact rejection: Amplitude threshold > 150 µV")

print(f"\n4️⃣ Transfer Learning Approach:")
print(f"   Stage 1: Pretrain CNN encoder on LEMON (unsupervised/contrastive)")
print(f"   Stage 2: Extract robust EEG representations")
print(f"   Stage 3: Fine-tune CNN+LSTM on migraine dataset")
print(f"   Stage 4: 3-class classification (Control, Aura, Non-Aura)")

print(f"\n5️⃣ Next Steps:")
print(f"   → Create preprocessing module (src/lemon_preprocessor.py)")
print(f"   → Create windowed dataset builder (src/windowed_dataset_builder.py)")
print(f"   → Create transfer learning training pipeline")
print(f"   → Expected accuracy improvement: 75% → 88-92%")

print("\n" + "="*60)
print("✓ EDA Complete - Ready for Preprocessing Pipeline")
print("="*60)